# Optimise the analysis of an smFISH image

This notebook helps you choose the settings for analysing smFISH images with [Big-FISH](https://big-fish.readthedocs.io/en/stable/). You work on one image and see what each setting does to it: how the background is removed, which spots are counted, how dense regions are split into single spots, and which spots form clusters. The settings you settle on are kept in `config`. Later steps save them as a settings file, so that the batch command can analyse all your images in the same way.

**How to use it**

1. Run the cells from top to bottom: click a cell, then press Shift + Enter.
2. The cells that start with `# Values` are the only ones you change. Edit the values, run that cell, then run the cells below it again.
3. Leave all other cells as they are. If a cell reports a problem, read the message, change the value it names and run again from the values cell.

As supplied, the values open the small test image in the `tests/data` folder, so you can run the whole notebook once before pointing it at your own images. Installation is described in the [README](../README.md); if you are new to JupyterLab, start with the [JupyterLab guide](https://jupyterlab.readthedocs.io/en/stable/getting_started/starting.html).

## 2. Set-up

This cell loads the pipeline, shows which versions are installed and checks which device Cellpose will use for cell segmentation later on.

In [ ]:
%matplotlib inline
import gc
import warnings
from functools import partial

import bigfish
import yaml
from IPython.display import display

import bigfish_pypeline
from bigfish_pypeline import batch, detection, io, notebook, outputs, qc, segmentation, steps
from bigfish_pypeline.config import OMERO_HOST, add_channel, new_config, update_channel, update_config


warnings.filterwarnings("ignore", message=".*mean_intensity.*")  # a notice from Big-FISH that needs no action


notebook.show_problems_plainly()  # problems with your values appear as one short message

config = None            # the settings, built in step 5 and kept for every channel
conn = None              # the OMERO session, if the image comes from OMERO
image = filtered = None  # the channel in memory (one at a time)
tables_by_channel = {}   # the spot and cluster tables of each channel tuned so far

device, device_report, device_warnings = segmentation.choose_device("auto")
print(f"bigfish-pypeline {bigfish_pypeline.__version__}, Big-FISH {bigfish.__version__}")
print(device_report)
for warning in device_warnings:
    print("Note:", warning)

## 3. Choose the image

Say where the image is. For a **local** image, give the folder and, if you like, some text that the file name contains; the cell below the values lists the matching images and you pick one by its number (the first is 0). For an image on **OMERO**, give the server, your user name and the image's ID number; the log-in cell then asks for your password in a box. The password is not shown on screen and is never stored or saved.

`cell_results` is the first of the two preset choices: switch it on if you want the spots counted in each cell, which needs a cell segmentation in the later steps. The second choice, the assay type, is made for each channel in step 5.

In [ ]:
# Values for step 3: edit these, then run this cell and the ones below it.
image_source = "local"                        # "local" for a folder on this computer, "omero" for an image on the OMERO server
cell_results = True                           # True to count the spots in each cell (needs a cell segmentation); False for the whole image
# Local folder (used when image_source is "local")
folder = "../tests/data"                      # the folder holding your images; ".." means the folder above this notebook
name_filter = "sypN_DAPI_Phal_jim-exon_crop"  # only images whose names contain this text are listed; "" lists them all
extension = ".tif"                            # ".tif" or ".tiff"
image_number = 0                              # the number of the image to open, from the list shown below the cell
# OMERO (used when image_source is "omero")
omero_host = ""                               # the server address; "" uses the lab server
omero_port = 4064                             # usually 4064
omero_username = ""                           # your OMERO user name
omero_image_id = 0                            # the ID number of the image, as shown in OMERO

In [ ]:
if image_source == "local":
    images = io.list_images(folder, name_filter, extension)
    print(io.numbered_images(images))
else:
    print("The image comes from OMERO, so there is no list of local images.")

In [ ]:
if image_source == "omero":
    from bigfish_pypeline import omero_tools

    conn = omero_tools.connect_to_omero(omero_username, omero_host or OMERO_HOST, omero_port)
else:
    print("The image is in a local folder, so there is no OMERO log-in.")

## 4. Image information

The cell below opens the image and lists what the file says about itself: the size, the order of its axes (Z for planes, C for channels, Y and X for the picture), the pixel size and the distance between planes. A value the image does not give is marked, and you enter it yourself.

Copy these into the values cell that follows, or correct them if you know better. The microscope values tell the pipeline how big a spot should look.

In [ ]:
if image_source == "local":
    image_path = images[image_number]
    info = io.get_image_info(image_path)
    image_name = io.output_name(image_path)
else:
    info = io.get_omero_image_info(conn, omero_image_id)
    image_name = info.output_name
suggestion = io.suggest_image_settings(info)
print(io.describe_suggestion(suggestion))

In [ ]:
# Values for step 4: edit these, then run this cell and the ones below it.
layout = "ZCYX"               # the order of the axes: "ZCYX", "CZYX", "ZYX", "CYX" or "YX"; see "Layout" above
voxel_size_yx_nm = 65         # pixel size in nm; see "Pixel size in x and y" above
voxel_size_z_nm = 200         # distance between planes in nm; see "Distance between planes (z)" above; not used for flat images
microscope_type = "confocal"  # "confocal", "widefield" or "nipkow" (spinning disk)
numerical_aperture = 1.45     # NA of the objective, usually 0.9 to 1.5
refractive_index = 1.45       # of the immersion medium, about 1.33 (water) to 1.52 (oil)
show_previews = True          # True to preview every channel; False to skip it (slow for large images from OMERO)

The cell below checks that the image can be read with this layout, then shows every channel as a maximum projection: at each position, the brightest value over all planes. Each channel has its own contrast, so compare shapes and not brightness. Channel numbers count from 0; note the number of your FISH channel for step 5.

In [ ]:
if image_source == "local":
    io.check_image(image_path, layout)
    read_channel = partial(io.read_channel, image_path, layout)
else:
    read_channel = partial(io.read_omero_channel, conn, omero_image_id, layout)
image_settings = {
    "axes": layout,
    "voxel_size_nm": {"yx": voxel_size_yx_nm, "z": voxel_size_z_nm if "Z" in layout else None},
}
microscope = {"type": microscope_type, "NA": numerical_aperture, "RI": refractive_index}
if config is not None:
    config = update_config(config, {"image": image_settings, "microscope": microscope})
if show_previews:
    projections = [qc.project_for_detection(read_channel(number)) for number in range(io.image_counts(info, layout)[1])]
    display(qc.draw_channel_previews(projections, suggestion.channel_names, image_name))
else:
    print("Previews are switched off.")

## 5. FISH channel

Choose the FISH channel to optimise. Only one channel is kept in memory at a time (a full-size channel with its filtered copies takes about 5 GB), so the cell below lets go of the previous channel before it reads the next.

The wavelengths are those of your fluorophore. The assay type is the second preset choice: `smFISH` switches on the splitting of dense regions (decomposition) and clusters; `amplified` switches both off, as is usual for amplified FISH such as HCR or RNAscope. You can still change each switch in step 8.

In [ ]:
# Values for step 5: edit these, then run this cell and the ones below it.
channel_index = 2        # the channel number, counted from 0, as in the previews above
channel_name = "jim-exon"  # your own name for this channel; it can stay empty
excitation_nm = 570      # excitation wavelength of the fluorophore, in nm
emission_nm = 610        # emission wavelength of the fluorophore, in nm
assay_type = "smFISH"    # "smFISH", or "amplified" for amplified FISH such as HCR or RNAscope

In [ ]:
channel_settings = {
    "index": channel_index, "name": channel_name, "ex": excitation_nm, "em": emission_nm, "assay_type": assay_type,
}
if config is None:
    config = new_config(layout, image_settings["voxel_size_nm"], [channel_settings], cell_results, microscope)
elif channel_index not in [channel.index for channel in config.channels]:
    config = add_channel(config, channel_settings)
else:
    config = update_channel(config, channel_index, {"name": channel_name, "ex": excitation_nm, "em": emission_nm})
channel = config.channel(channel_index)

image = filtered = None  # let go of the previous channel before reading the next
gc.collect()
image = read_channel(channel_index)
print(f"Channel {channel_index} read: {image.shape} (planes, y, x), {image.dtype}")

## 6. Background and LoG filter

First the background is subtracted from each plane, then a Laplacian of Gaussian (LoG) filter is applied. The filter makes spots stand out as bright peaks, and spots are found in this LoG picture. Its values (the "LoG value") are what the threshold in step 7 is compared with.

The radius is how far the background method looks around each pixel. It should be larger than a spot (the spot radius is printed below) and smaller than the blotches you want to remove. `rolling_ball` is the usual choice for new analyses; `tophat` reproduces older results.

In [ ]:
# Values for step 6: edit these, then run this cell and the ones below it.
background_method = "rolling_ball"  # "rolling_ball" or "tophat"
background_radius_px = 5            # radius in pixels, 1 or more; larger keeps more of the signal, smaller removes more

In [ ]:
config = update_channel(config, channel_index, {
    "background": {"method": background_method, "radius_px": background_radius_px},
})
channel = config.channel(channel_index)
radius = steps.spot_radius(config, channel)
filtered = None
filtered = steps.filter_image(config, channel, image, radius["spot_radius_px"])
log_mip = qc.project_for_detection(filtered["rna_log"])
print("Spot radius in pixels (planes, y, x):", tuple(round(float(value), 2) for value in radius["spot_radius_px"]))

The picture below is the LoG maximum projection. Drag to move around, scroll to zoom, and hover to read the position and the value of a pixel. Spots should be small, bright points on a dark, even background. If large patches stay bright, make the radius smaller; if spots look faint or are missing, try a larger one.

In [ ]:
qc.draw_log_view(log_mip, image_name=image_name, channel=channel_index, channel_name=channel.name)

To choose a threshold, it helps to know how bright a spot is compared with the background. Draw a line across a few spots and the values along it are plotted; the picture above shows positions as (y, x) in pixels. The histogram counts the pixels of each LoG value on a logarithmic scale. You can also save the LoG picture as a TIFF file to open in ImageJ or Fiji.

In [ ]:
# Values for step 6 (pictures): edit these, then run this cell.
profile_start = (186, 150)           # (y, x) in pixels, where the line starts
profile_end = (186, 220)             # (y, x) in pixels, where the line ends
save_log_picture = False             # True to save the LoG picture as a TIFF file for ImageJ or Fiji
log_picture_file = "log-picture.tif"  # the file to write, next to this notebook

In [ ]:
display(qc.plot_line_profile(log_mip, profile_start, profile_end))
display(qc.draw_log_histogram(log_mip, image_name=image_name, channel=channel_index, channel_name=channel.name))
if save_log_picture:
    qc.write_log_projection(log_mip, log_picture_file, config.image.voxel_size_nm.yx)
    print("Saved", log_picture_file)

## 7. Threshold

A spot is a peak of the LoG picture that is higher than the threshold. There are three ways to set it; as supplied, the notebook uses `auto`:

- `auto`: Big-FISH finds a threshold for each image with the elbow method. `lower_limit` stops the value falling below a floor you choose, which protects images with noise and no real spots.
- `manual`: you give the value in `threshold_value`. Use the line profile and the histogram above to see how high spots and background are.
- `calibrated`: the median of the automatic values of several images is used for all of them. Set `calibration_filter` to text that the names of those images contain, and run the optional calibration cell below.

`bailout_max_spots` skips a channel in the batch when it finds more spots than this, as a guard against images with noise or debris. Leave it `None` to count every spot.

The cell after the values only writes them into `config`.

In [ ]:
# Values for step 7: edit these, then run this cell and the ones below it.
threshold_mode = "auto"     # "manual", "auto" or "calibrated"
threshold_value = None      # manual mode only: the threshold, a LoG value such as 18 (see the line profile and histogram); None in the other modes
lower_limit = None          # auto and calibrated modes: the threshold is never lower than this; None for no limit
calibration_filter = ""     # calibrated mode: text that the names of the calibration images contain
bailout_max_spots = None    # skip the channel when more spots than this are found; None for no limit
run_calibration = False     # True to run the optional calibration cell below (local images only)

In [ ]:
config = update_channel(config, channel_index, {
    "threshold": {
        "mode": threshold_mode, "value": threshold_value, "lower_limit": lower_limit,
        "calibration_filter": calibration_filter,
    },
    "bailout": {"enabled": bailout_max_spots is not None, "max_spots": bailout_max_spots},
})
channel = config.channel(channel_index)
print(f"Threshold mode: {channel.threshold.mode}")

**Optional: calibration.** Run this only for the `calibrated` mode. It finds the automatic threshold of every image in the folder whose name contains `calibration_filter`, takes their median and uses it for this channel. Each image is read once, so with large images it takes a few minutes and needs memory for one more channel. The plot shows the value of each image and the median used. Images that cannot be used are marked and left out of the median.

In [ ]:
if not run_calibration:
    print("Calibration is switched off (run_calibration = False).")
elif image_source != "local":
    print("Calibration here needs images in a local folder. With OMERO images the batch calibrates for you.")
else:
    paths = io.list_images(folder, calibration_filter, extension)
    results = [batch.calibration_threshold(config, io.output_name(path), path, channel_index) for path in paths]
    value, table = detection.calibrate_threshold(results)
    rows = [
        dict(image=row["image"], channel=channel_index, auto_threshold=row["threshold"],
             elbow_threshold=result["elbow_threshold"], used=row["used"], reason=row["reason"])
        for row, result in zip(table, results)
    ]
    display(qc.draw_calibration_plot(
        outputs.make_calibration_table(rows), {channel_index: value},
        {channel_index: channel.threshold.lower_limit}, {channel_index: channel.name},
    ))
    known = config.resolved.thresholds if config.resolved is not None else {}
    config = config.with_resolved([path.name for path in paths], {**known, channel_index: value})
    print(f"Calibrated threshold of channel {channel_index}: {value:.1f}, the median of {len(paths)} image(s).")
    if channel.threshold.mode != "calibrated":
        print('It is used when threshold_mode is "calibrated".')

The cell below finds the spots with this threshold and says which threshold was used, with the automatic value of this image beside it. In the **elbow plot** the curve falls steeply and then flattens; the automatic threshold is at the bend. In the **histogram** the red line shows where the threshold cuts the LoG values. In the **detection plot** the orange circles mark the spots on the maximum projection of the background-subtracted channel.

What to look for: circles on empty background mean the threshold is too low, so raise it; bright spots without a circle mean it is too high, so lower it. Change the values above and run the cells from there again until the circles sit on the spots you would count by eye.

In [ ]:
detected = steps.detect(config, channel, filtered["rna_log"], radius["spot_radius_px"], elbow=True)
mode = channel.threshold.mode
limit = None if mode == "manual" else channel.threshold.lower_limit
print("\n".join(qc.describe_threshold(detected, mode, limit)))
reason = steps.bailout_reason(config, channel, detected["spots"])
if reason:
    print("The batch would skip this channel:", reason)
elbow_plot = qc.draw_elbow_plot(detected, image_name, channel_index, mode, limit, channel.name)
if elbow_plot is not None:
    display(elbow_plot)
display(qc.draw_log_histogram(
    log_mip, detected["threshold"], image_name=image_name, channel=channel_index, channel_name=channel.name,
))
qc.draw_detection_plot(
    filtered["rna_mip"], detected["spots"], image_name, channel_index, radius["spot_radius_px"], channel_name=channel.name,
)

## 8. Dense regions, subpixel positions, clusters and measurements

Each of the four parts below can be switched on or off for the channel. A part that is switched off is skipped and the spots pass on unchanged. In the three values cells that offer `None`, `None` keeps the choice of the assay type from step 5; `True` or `False` overrides it.

### 8a. Decomposition of dense regions

In dense regions, such as a transcription site, spots overlap and are counted as one. Decomposition builds a reference spot, the typical single spot of this image, and uses it to work out how many spots a dense region holds. The reference spot should look like one compact blob, brightest in the middle plane.

In [ ]:
# Values for step 8a: edit these, then run this cell and the ones below it.
use_decomposition = None  # None keeps the assay type's choice; True or False switches decomposition on or off
alpha = 0.5               # 0 to 1: which spot is the reference; 0.5 is the median spot, higher means a brighter spot and fewer added spots
beta = 1                  # above 0: regions brighter than beta times the median spot are split; higher means fewer regions are split

In [ ]:
changes = {"alpha": alpha, "beta": beta}
if use_decomposition is not None:
    changes["enabled"] = use_decomposition
config = update_channel(config, channel_index, {"decomposition": changes})
channel = config.channel(channel_index)
decomposed = steps.decompose(config, channel, image_name, filtered["rna"], detected["spots"], radius["spot_radius_nm"])
spots = decomposed["spots_post_decomposition"]
if not channel.decomposition.enabled:
    print("Decomposition is switched off for this channel.")
else:
    print(f"Spots before decomposition: {len(detected['spots'])}; after: {len(spots)}.")
    if decomposed["warning"]:
        print("Note:", decomposed["warning"])
    if decomposed["reference_spot_for_qc"]:
        display(qc.draw_reference_spot(decomposed["reference_spot"], image_name, channel_index, channel.name))

### 8b. Subpixel fitting

Spots are first placed at whole pixels. Subpixel fitting moves each one to its position to a fraction of a pixel, which matters for distances between spots. It takes longer, so it is off unless you switch it on.

In [ ]:
# Values for step 8b: edit these, then run this cell and the ones below it.
use_subpixel = False  # True to fit the positions of the spots to a fraction of a pixel

In [ ]:
config = update_channel(config, channel_index, {"subpixel": {"enabled": use_subpixel}})
channel = config.channel(channel_index)
fitted = steps.fit_subpixel(config, channel, filtered["rna"], spots, radius["spot_radius_nm"])
if not channel.subpixel.enabled:
    print("Subpixel fitting is switched off for this channel.")
else:
    print(f"Positions fitted to a fraction of a pixel for {len(spots)} spots.")

### 8c. Clusters

Spots that lie close together form a cluster, for example an mRNA granule. Spots closer than the cluster radius are connected, and a group with at least the minimum number of spots is a cluster. A larger radius or a smaller minimum gives more and bigger clusters.

In [ ]:
# Values for step 8c: edit these, then run this cell and the ones below it.
use_clusters = None      # None keeps the assay type's choice; True or False switches clusters on or off
cluster_radius_nm = 500  # spots closer than this, in nm, are connected; larger gives bigger clusters
cluster_min_spots = 4    # at least this many connected spots (2 or more) make a cluster; smaller gives more clusters

In [ ]:
changes = {"radius_nm": cluster_radius_nm, "min_spots": cluster_min_spots}
if use_clusters is not None:
    changes["enabled"] = use_clusters
config = update_channel(config, channel_index, {"clusters": changes})
channel = config.channel(channel_index)
clustered = steps.find_clusters(config, channel, spots, fitted["spots_subpixel"])
if not channel.clusters.enabled:
    print("Clusters are switched off for this channel.")
else:
    in_clusters = int((clustered["cluster_id"] >= 0).sum())
    print(f"{len(clustered['clusters'])} clusters, holding {in_clusters} of the {len(spots)} spots.")

### 8d. Measurements

Two measurements can be added to each spot. The signal-to-noise ratio compares the spot with the ring of pixels around it. The intensity is the LoG value at the spot, which is the value the threshold was compared with.

In [ ]:
# Values for step 8d: edit these, then run this cell and the ones below it.
measure_snr = False        # True to add the signal-to-noise ratio of each spot
measure_intensity = False  # True to add the LoG value at each spot

In [ ]:
config = update_channel(config, channel_index, {"measurements": {"snr": measure_snr, "intensity": measure_intensity}})
channel = config.channel(channel_index)
measured = steps.measure(config, channel, filtered["rna"], filtered["rna_log"], spots, radius["spot_radius_px"])
if not (channel.measurements.snr or channel.measurements.intensity):
    print("Measurements are switched off for this channel.")
else:
    print("Measured for each spot:", ", ".join(measured))

### The result for this channel

The detection plot now shows the final spots as orange circles and the clusters as cyan outlines. The tables below are what the batch would save for this image and channel; the first rows are shown. Run this cell again after any change above.

In [ ]:
assigned = steps.assign_to_cells(config, channel, None, spots, clustered)
tables = steps.make_tables(config, channel, image_name, spots, fitted, clustered, measured, assigned)
tables_by_channel[channel_index] = tables
cluster_table = clustered["clusters"]
display(qc.draw_detection_plot(
    filtered["rna_mip"], steps.final_positions(spots, fitted), image_name, channel_index, radius["spot_radius_px"],
    clusters=None if cluster_table is None else cluster_table[:, :steps.get_ndim(config)],
    cluster_radius_px=channel.clusters.radius_nm / config.image.voxel_size_nm.yx, channel_name=channel.name,
))
if channel.clusters.enabled:
    display(tables["clusters"].head())
tables["spots"].head()

## 9. Further channels

To tune another FISH channel, go back to step 5: change the values for the channel number, name and wavelengths, then run the cells again from step 5 down to the end of step 8. The previous channel's images are let go and the new channel is read. The settings of every channel you have tuned stay in `config`, and the tables of each are kept in `tables_by_channel`.

The settings chosen so far are printed below.

In [ ]:
chosen = ("image", "microscope", "channels", "resolved")
print(yaml.safe_dump({name: value for name, value in config.to_dict().items() if name in chosen}, sort_keys=False))

The next steps of this notebook (cell segmentation, assigning spots to cells, saving the results of this image and saving the settings file for the batch) follow. Until they are added, `config` holds the settings of each channel tuned so far.

### Closing the OMERO session

Run this last cell when you have finished. It ends your OMERO session; for a local image it does nothing.

In [ ]:
if conn is not None:
    conn.close()
    conn = None
    print("The OMERO session is closed.")
else:
    print("There is no OMERO session to close.")